In [ ]:
import scripts.helpers as helpers

1. Step 1 - Create unified labels file -> in parquete format
2. Step 2 - Unify each SWC tree with it's labels
3. Step 3 - Simplify SWC tree
4. Step 4 - Calculate clumpiness for each node in the tree, for each labels combination
5. Step 5 - Insert the clumpiness score  into the simplified SWC file

---
# Step 1 - Create unified labels file -> in parquete format

---
# Step 2 - Unify each SWC tree with it's labels

---
# Step 3 - Simplify SWC tree

---
# Step 4 - Calculate clumpiness for each node in the tree, for each labels combination

---
# Step 5 - Insert the clumpiness score  into the simplified SWC file

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [34]:
import os
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from scripts.preprocessing import simplify_swc_topology


# Type data located in the 

path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
swc_labels = pd.read_feather(path_swc_labels)

required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

In [ ]:
#  0. Get the list of required swc files
#  1. Load labels parquet
#  2. Import the swc file
#  3. simplify swc file
#  4. attach synapse labels + neuron type
#  5. save simplified file
#  6. convert to json for find-clumpiness
#  7. save json file
#  8. calculate clumpiness for each internal node
#  9. attach results to the labeled swc file
# 10. save results.

In [ ]:
#########################
#  1. Load labels parquet
# Parquet labels path
prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

# Load exactly the labels of the example swc file
parquet_labels = pl.scan_parquet(prquet_labels_path)

# Only the relevnt column in the parquet file
neuron_id = 720575940609102805
labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(neuron_id)).collect().to_pandas()


#########################
#  2. Import the swc file
neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{neuron_id}.swc")
neuron_swc = pd.read_csv(neuron_path, 
                         comment='#', 
                         header=None, 
                         sep=r'\s+', 
                         names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


#######################
#  3. simplify swc file
simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{720575940609102805}", save_csv=False)


########################################
#  4. attach synapse labels + neuron type
swc_labeled = pd.merge(left=simple_swc, 
                       right=labels_parquet[["node_id", "type"]], 
                       left_on="node_id", 
                       right_on="node_id", how="left")


##########################
#  5. save simplified file